# Calcium transient fitting

In [ ]:
using ModelingToolkit
using CurveFit
using OrdinaryDiffEq
using Plots
using CaMKIIModel
using CaMKIIModel: second, μM

## Setup the ODE system

In [ ]:
stimstart = 0.0second
stimend = 100.0second
tend = stimend
@time "Building ODE system" @mtkcompile sys = build_neonatal_ecc_sys()
@time "Building ODE problem" prob = ODEProblem(sys, [], tend)

In [ ]:
@unpack Istim = sys
callback = build_stim_callbacks(Istim, stimend; period=1second, starttime=stimstart)
@time "Solving ODE problem" sol = solve(prob, KenCarp47(); callback)

## Fit the calcium transient curve

Using rational polynomial fit

In [ ]:
ts = range(0.0, 1second, length=101)
cai = sol(ts .+ 99second, idxs=sys.Cai_mean).u

In [ ]:
prob = CurveFitProblem(ts, cai)
@time fit = solve(prob, RationalPolynomialFitAlgorithm(4, 4))
plot(ts, cai, label="Data", line=:solid)
plot!(ts, fit.(ts), label="Fit", line=:dash)
plot!(xlabel="Time (ms)", ylabel="Ca (μM)", title="Calcium transient fitting")

In [ ]:
println("Numerator coefficients: ", fit.u[1:5])
println("Denominator coefficients: ", vcat(1.0, fit.u[6:end]))
println("RMSE: ", mse(fit) |> sqrt)